In [33]:
read_data <- function(path) {
  d <- read.csv(path, stringsAsFactors = FALSE)
  d <- d[, c("Azimuth", "Lat", "Lon", "GpsLat", "GpsLon")]
  d <- d[!is.na(d$Azimuth),]
  d$punto <- seq_len(nrow(d))
  d
}

In [34]:
d <- read_data("resection_log_golden.csv")

resection <- function(az, x, y, add = rep(TRUE, length(az))) {
  az <- az[add]
  x <- x[add]
  y <- y[add]
  a <- az * rad
  nx <- cos(a)
  ny <- -sin(a)
  b <- nx * x + ny * y
  A <- matrix(c(sum(nx^2), sum(nx * ny), sum(nx * ny), sum(ny^2)), 2, 2)
  p <- solve(A, c(sum(nx * b), sum(ny * b)))
  rex <- nx * (p[1] - x) + ny * (p[2] - y)
  list(east = p[1], north = p[2], error = sqrt(sum(p^2)))
}


In [35]:
RADIUS_EARTH <- 6371008.0
rad <- pi / 180

In [36]:
to_degrees <- function(x, y, origin) {
  c(
    lat = origin[["lat"]] + y / (rad * RADIUS_EARTH),
    lon = origin[["lon"]] +
      x / (rad * RADIUS_EARTH * cos(origin[["lat"]] * rad))
  )
}

to_meters <- function(lat, lon, origin) {
  x <- (lon - origin[["lon"]]) *
    rad *
    RADIUS_EARTH *
    cos(origin[["lat"]] * rad)
  y <- (lat - origin[["lat"]]) * rad * RADIUS_EARTH
  cbind(x = x, y = y)
}


In [38]:
run <- function(table, origin, k = 3, daz_val = 1, dpos_val = 0.0001) {
  t <- table[seq_len(k),]
  n <- nrow(t)

  row_resecton <- function(
    name,
    daz = rep(0, n),
    dpos = rep(0, n),
    add = rep(TRUE, n)
  ) {
    lat <- t$lat + dpos
    lon <- t$lon + dpos
    az <- t$az + daz
    xy <- to_meters(lat, lon, origin)
    s <- resection(az, xy[, "x"], xy[, "y"], add)
    sol <- to_degrees(s$east, s$north, origin)
    P <- ifelse(add, round(az, 2), NA)
    Q <- ifelse(add, sprintf("(%.6f, %.6f)", lat, lon), NA)
    out <- as.data.frame(
      c(
        list(scenario = name),
        setNames(as.list(P), paste0("P", 1:n)),
        setNames(as.list(Q), paste0("Q", 1:n))
      ),
      stringAsFactors = FALSE
    )

    out$Result1 <- sprintf("(%.7f, %.7f)", sol[["lat"]], sol[["lon"]])
    out$Result2 <- sprintf("(%.7f, %.7f)", origin[["lat"]], origin[["lon"]])
    out$error_origin_m <- s$error
    out
  }

  r <- row_resecton("BASE")
  for (i in 1:n) {
    d <- rep(0, n)
    d[i] <- daz_val
    r <- rbind(r, row_resecton(paste0("+1 Azimuth angle in P", i), daz = d))
  }
  r <- rbind(r, row_resecton("+1 Azimuth angle in all", daz = rep(daz_val, n)))
  for (i in 1:n) {
    d <- rep(0, n)
    d[i] <- dpos_val
    r <- rbind(r, row_resecton(paste0("+0.0001 lat/lon in P", i), dpos = d))
  }
  r <- rbind(r, row_resecton("+0.0001 lat/lon in all", dpos = rep(dpos_val, n)))
  if (n > 2) {
    for (i in 1:n) {
      r <- rbind(
        r,
        row_resecton(paste0("Without P", i), add = seq_len(n) != i)
      )
    }
  }
  r
}

In [63]:
origin <- c(lat = 48.7022484, lon = 2.1530186)
table <- data.frame(point = d$punto, az = d$Azimuth, lat = d$Lat, lon = d$Lon)

res <- run(table, origin, k = 5)

In [64]:
View(res)

scenario,P1,P2,P3,P4,P5,Q1,Q2,Q3,Q4,Q5,Result1,Result2,error_origin_m
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
BASE,167.86,265.2,39.63,2.7,127.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7022113, 2.1528518)","(48.7022484, 2.1530186)",12.9179635
+1 Azimuth angle in P1,168.86,265.2,39.63,2.7,127.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7022191, 2.1528866)","(48.7022484, 2.1530186)",10.2225197
+1 Azimuth angle in P2,167.86,266.2,39.63,2.7,127.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7020804, 2.1528671)","(48.7022484, 2.1530186)",21.7412454
+1 Azimuth angle in P3,167.86,265.2,40.63,2.7,127.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7022840, 2.1527586)","(48.7022484, 2.1530186)",19.4846565
+1 Azimuth angle in P4,167.86,265.2,39.63,3.7,127.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7022680, 2.1520939)","(48.7022484, 2.1530186)",67.8915214
+1 Azimuth angle in P5,167.86,265.2,39.63,2.7,128.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7024563, 2.1530552)","(48.7022484, 2.1530186)",23.2724256
+1 Azimuth angle in all,168.86,266.2,40.63,3.7,128.69,"(48.698253, 2.153803)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7024628, 2.1522576)","(48.7022484, 2.1530186)",60.7200794
+0.0001 lat/lon in P1,167.86,265.2,39.63,2.7,127.69,"(48.698353, 2.153903)","(48.700938, 2.129361)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7022191, 2.1528948)","(48.7022484, 2.1530186)",9.6529832
+0.0001 lat/lon in P2,167.86,265.2,39.63,2.7,127.69,"(48.698253, 2.153803)","(48.701039, 2.129461)","(48.712260, 2.165583)","(48.785958, 2.159018)","(48.679200, 2.198176)","(48.7022567, 2.1528461)","(48.7022484, 2.1530186)",12.6909443


In [55]:
library(ggplot2)
library(ggrepel)

AZUL <- "#2a78d6"
NARANJA <- "#eb6834"
VERDE <- "#1baf7a"
GRIS <- "#8a8a8a"
dir.create("charts", showWarnings = FALSE)


# ---------- datos comunes ----------
xy <- to_meters(table$lat, table$lon, origin)
pines <- data.frame(
  punto = paste0("P", seq_len(nrow(table))),
  x = xy[, "x"], y = xy[, "y"], az = table$az
)

pines$punto      # debe mostrar "P1" "P2" "P3"

parse_ll <- function(s) {
  n <- as.numeric(strsplit(gsub("[()]", "", s), ", ")[[1]])
  c(lat = n[1], lon = n[2])
}

sol_m <- t(sapply(res$Result1, function(s) {
  p <- parse_ll(s)
  to_meters(p[["lat"]], p[["lon"]], origin)
}))
dimnames(sol_m) <- list(NULL, c("x", "y"))
base <- sol_m[1,]


# ---------- 1. MAPA EN METROS ----------
L <- 4000
LBL_REAL <- "True direction (origin → point)"
LBL_AZ <- "Measured azimuth line"

rayos <- transform(pines, dx = sin(az * pi / 180), dy = cos(az * pi / 180))
rayos$x0 <- rayos$x - L * rayos$dx
rayos$y0 <- rayos$y - L * rayos$dy
rayos$x1 <- rayos$x + L * rayos$dx
rayos$y1 <- rayos$y + L * rayos$dy

radios <- 300 + 200 * seq_len(nrow(pines))
arcos <- do.call(rbind, lapply(seq_len(nrow(pines)), function(i) {
  a <- seq(0, pines$az[i], length.out = 80) * pi / 180
  data.frame(punto = pines$punto[i], x = radios[i] * sin(a), y = radios[i] * cos(a))
}))

# Etiqueta al final de cada arco
etq <- data.frame(
  x = radios * sin(pines$az * pi / 180),
  y = radios * cos(pines$az * pi / 180),
  txt = sprintf("%s: %.1f°", pines$punto, pines$az),
  az = pines$az
)

todos_x <- c(pines$x, 0, base[["x"]])
todos_y <- c(pines$y, 0, base[["y"]])
mx <- 0.12 * diff(range(todos_x))
my <- 0.12 * diff(range(todos_y))

g1 <- ggplot() +
  geom_segment(data = rayos, aes(x = x0, y = y0, xend = x1, yend = y1, linetype = LBL_AZ, colour = LBL_AZ),
               linewidth = 0.5) +
  geom_segment(
    data = pines,
    aes(
      x = 0,
      y = 0,
      xend = x,
      yend = y,
      linetype = LBL_REAL,
      colour = LBL_REAL
    ),
    linewidth = 0.8
  ) +
  geom_segment(
    aes(x = 0, y = 0, xend = base[["x"]], yend = base[["y"]]),
    colour = NARANJA,
    linewidth = 1.2,
    arrow = arrow(length = unit(0.25, "cm"))
  ) +
  # línea de referencia Norte
  annotate("segment", x = 0, y = 0, xend = 0, yend = max(radios) + 150,
           colour = "black", linewidth = 0.4) +
  annotate("text", x = 0, y = max(radios) + 230, label = "N",
           fontface = "bold", size = 3.6) +

  # arcos del ángulo azimut
  geom_path(data = arcos, aes(x, y, group = punto),
            colour = NARANJA, linewidth = 0.8,
            arrow = arrow(length = unit(0.12, "cm"), type = "closed")) +
  geom_text(data = etq,
            aes(x, y, label = txt,
                hjust = ifelse(sin(az * pi / 180) >= 0, -0.12, 1.12),
                vjust = ifelse(cos(az * pi / 180) >= 0, -0.35, 1.35)),
            colour = NARANJA, size = 3.1, fontface = "bold") +
  geom_point(data = pines, aes(x, y), size = 3) +
  geom_text_repel(data = pines, aes(x, y, label = punto), fontface = "bold") +
  geom_point(aes(x = 0, y = 0), shape = 8, size = 3.5) +
  geom_point(aes(x = base[["x"]], y = base[["y"]]), colour = AZUL, size = 3.5) +
  annotate("label", x = -1900, y = 1000,
           label = sprintf("Solution (BASE)\nerror %.0f m", res$error_origin_m[1]),
           colour = AZUL, fontface = "bold", size = 3.5, fill = "white", border.colour = NA) +
  annotate("segment", x = -1250, y = 850,
           xend = base[["x"]] - 25, yend = base[["y"]] + 25,
           colour = AZUL, linewidth = 0.4) +
  annotate("label", x = -900, y = 1450, label = "Origin",
           size = 3.5, fill = "white", border.colour = NA) +
  annotate("segment", x = -700, y = 1350, xend = -25, yend = 25,
           colour = "black", linewidth = 0.3) +

  # UNA sola leyenda: colour y linetype con los mismos nombres
  scale_colour_manual(values = setNames(c(GRIS, VERDE), c(LBL_REAL, LBL_AZ)), name = NULL) +
  scale_linetype_manual(values = setNames(c("solid", "dashed"), c(LBL_REAL, LBL_AZ)), name = NULL) +
  coord_fixed(xlim = c(-2500, 3900), ylim = c(-3100, 1900)) +
  labs(
    title = "Map in meters: azimith angles (from north, clockwise)",
    x = "East (m)",
    y = "North (m)"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom", plot.title = element_text(face = "bold"))

[1] "P1" "P2" "P3" "P4" "P5"

In [56]:
ggsave("charts/1_mapa_metros.png", g1, width = 9, height = 7, dpi = 150)


In [65]:

pines$rumbo_real <- (atan2(pines$x, pines$y) * 180 / pi) %% 360
pines$error_ang <- ((pines$az - pines$rumbo_real + 180) %% 360) - 180   # entre -180 y 180

g2 <- ggplot(pines, aes(punto, error_ang)) +
  geom_hline(yintercept = 0, colour = "black") +
  geom_col(fill = AZUL, width = 0.55) +
  geom_text(aes(label = sprintf("%+.1f°", error_ang), vjust = ifelse(error_ang >= 0, -0.4, 1.4)), fontface = "bold") +
  scale_y_continuous(expand = expansion(mult = 0.15)) +
  labs(title = "Error angular por punto (azimut medido − rumbo real)",
       subtitle = "negativo = apuntaste a la izquierda del pin; positivo = a la derecha", x = NULL, y = "error (grados)") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"), panel.grid.major.x = element_blank())
ggsave("charts/2_error_angular.png", g2, width = 7, height = 5, dpi = 150)


In [66]:
# ---------- 3. SENSIBILIDAD ----------
sens <- data.frame(scenario = res$scenario, x = sol_m[, "x"], y = sol_m[, "y"], error = res$error_origin_m)
sens$delta_m <- sqrt((sens$x - base[["x"]])^2 + (sens$y - base[["y"]])^2)
sens$grupo <- ifelse(grepl("Azimuth", sens$scenario), "Azimut change",
                     ifelse(grepl("lat/lon", sens$scenario), "Point change", "Without a Point"))
sens <- sens[sens$scenario != "BASE",]
sens$scenario <- factor(sens$scenario, levels = rev(sens$scenario))

g3 <- ggplot(sens, aes(delta_m, scenario, colour = grupo)) +
  geom_segment(aes(x = 0.5, xend = delta_m, yend = scenario), linewidth = 0.8) +
  geom_point(size = 2.4) +
  geom_text(aes(label = sprintf("%.2f m", delta_m)), hjust = -0.35, size = 3.5, show.legend = FALSE) +
  scale_x_log10(limits = c(0.5, max(sens$delta_m) * 4),
                breaks = c(0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000),
                labels = c("0.1", "0.5", "1", "5", "10", "50", "100", "500", "1000")) +
  scale_colour_manual(values = c("Azimut change" = AZUL, "Point change" = NARANJA, "Without a Point" = VERDE), name = NULL) +
  labs(title = "Sensitivity: how far the solution shifts from the BASE/origin",
       x = "Displacement of the solution (m, logarithmic scale)", y = NULL) +
  theme_minimal(base_size = 11) +
  theme(legend.position = "bottom", plot.title = element_text(face = "bold"), panel.grid.minor = element_blank())
ggsave("charts/2_sensibilidad.png", g3, width = 9, height = 6, dpi = 150)
print(pines[, c("punto", "az", "rumbo_real", "error_ang")]); print(sens[, c("scenario", "delta_m", "error")])


  punto     az rumbo_real    error_ang
1    P1 167.86 172.618473 -4.758473014
2    P2 265.20 265.204337 -0.004337453
3    P3  39.63  39.634647 -0.004646808
4    P4   2.70   2.708218 -0.008217726
5    P5 127.69 127.716921 -0.026921175
                  scenario    delta_m      error
2   +1 Azimuth angle in P1  2.6970811 10.2225197
3   +1 Azimuth angle in P2 14.5986761 21.7412454
4   +1 Azimuth angle in P3 10.5890737 19.4846565
5   +1 Azimuth angle in P4 55.9751863 67.8915214
6   +1 Azimuth angle in P5 31.0640354 23.2724256
7  +1 Azimuth angle in all 51.8028403 60.7200794
8    +0.0001 lat/lon in P1  3.2726061  9.6529832
9    +0.0001 lat/lon in P2  5.0655564 12.6909443
10   +0.0001 lat/lon in P3  0.5992679 13.1531994
11   +0.0001 lat/lon in P4  2.3380153 10.8285693
12   +0.0001 lat/lon in P5  5.6456217  9.6208777
13  +0.0001 lat/lon in all 13.3228362  8.5416462
14              Without P1 12.7425434  0.7693103
15              Without P2  3.0126171 13.9535252
16              Without P3  4.8